In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [4]:
# Load data
train_path = "data/train.csv"
test_path = "data/test.csv"
sample_sub_path = "data/sample_submission.csv"

In [5]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

In [6]:
target_col = "diagnosed_diabetes"
id_col = "id"

In [7]:
# Split features and target
X = train_df.drop(columns=[target_col]).copy()
y = train_df[target_col].copy()
X_test = test_df.copy()

In [8]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

num_cols = [col for col in num_cols if col in X.columns]
cat_cols = [col for col in cat_cols if col in X.columns]

X = X[num_cols + cat_cols].copy()
X_test = X_test[num_cols + cat_cols].copy()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)
print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

Numeric columns: ['id', 'age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history']
Categorical columns: ['gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status']
X shape: (700000, 25)
X_test shape: (300000, 25)


In [9]:
# Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ],
    verbose_feature_names_out=False
)

In [10]:
# LightGBM pipeline
lgbm_model = LGBMClassifier(
    verbosity=-1,
    objective="binary",
    boosting_type="gbdt",
    random_state=42,
    n_jobs=-1
)

lgbm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", lgbm_model)
])

# Hyperparameter search space
param_dist = {
    "model__num_leaves": [31, 50, 64, 80],
    "model__max_depth": [-1, 5, 6, 7],
    "model__learning_rate": [0.03, 0.05, 0.07],
    "model__n_estimators": [300, 500, 700],
    "model__min_child_samples": [20, 40, 60],
    "model__subsample": [0.7, 0.8, 0.9],
    "model__colsample_bytree": [0.7, 0.8, 0.9],
    "model__reg_alpha": [0.0, 0.5, 1.0],
    "model__reg_lambda": [0.0, 0.5, 1.0]
}

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=param_dist,
    n_iter=12,
    scoring="roc_auc",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

# Run search
random_search.fit(X, y)

print("Best CV score:", random_search.best_score_)
print("Best params:")

for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits


/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning

Best CV score: 0.7256695733960667
Best params:
  model__subsample: 0.9
  model__reg_lambda: 0.5
  model__reg_alpha: 1.0
  model__num_leaves: 80
  model__n_estimators: 500
  model__min_child_samples: 60
  model__max_depth: -1
  model__learning_rate: 0.03
  model__colsample_bytree: 0.7


In [12]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

xgb_param_dist = {
    "model__max_depth": [3, 4, 5, 6],
    "model__learning_rate": [0.03, 0.05, 0.07],
    "model__n_estimators": [300, 500, 700],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.7, 0.8, 0.9],
    "model__colsample_bytree": [0.7, 0.8, 0.9],
    "model__reg_alpha": [0.0, 0.5, 1.0],
    "model__reg_lambda": [0.5, 1.0, 2.0]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    refit=True
)

xgb_search.fit(X, y)

print("XGBoost Best CV score:", xgb_search.best_score_)
print("XGBoost Best params:")
for k, v in xgb_search.best_params_.items():
    print(f"  {k}: {v}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
XGBoost Best CV score: 0.7261790067494215
XGBoost Best params:
  model__subsample: 0.8
  model__reg_lambda: 2.0
  model__reg_alpha: 0.5
  model__n_estimators: 700
  model__min_child_weight: 5
  model__max_depth: 5
  model__learning_rate: 0.07
  model__colsample_bytree: 0.8


In [13]:
cat_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=0
)

cat_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", cat_model)
])

cat_param_dist = {
    "model__depth": [4, 5, 6, 7],
    "model__learning_rate": [0.03, 0.05, 0.07],
    "model__iterations": [300, 500, 700],
    "model__l2_leaf_reg": [1, 3, 5, 7, 9],
    "model__subsample": [0.7, 0.8, 0.9]
}

cat_search = RandomizedSearchCV(
    estimator=cat_pipeline,
    param_distributions=cat_param_dist,
    n_iter=8,
    scoring="roc_auc",
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    refit=True
)

cat_search.fit(X, y)

print("CatBoost Best CV score:", cat_search.best_score_)
print("CatBoost Best params:")
for k, v in cat_search.best_params_.items():
    print(f"  {k}: {v}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


CatBoost Best CV score: 0.7256277814335481
CatBoost Best params:
  model__subsample: 0.9
  model__learning_rate: 0.07
  model__l2_leaf_reg: 5
  model__iterations: 700
  model__depth: 7


In [14]:
# Best model
best_lgbm_pipeline = random_search.best_estimator_
best_xgb_pipeline = xgb_search.best_estimator_
best_cat_pipeline = cat_search.best_estimator_

# Predict test
lgb_pred = best_lgbm_pipeline.predict_proba(X_test)[:, 1]
xgb_pred = best_xgb_pipeline.predict_proba(X_test)[:, 1]
cat_pred = best_cat_pipeline.predict_proba(X_test)[:, 1]

/Users/heosunghak/miniforge3/envs/titanic/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [15]:
# Save individual submissions

target_submission_col = sample_sub.columns[1]

submission_lgb = sample_sub.copy()
submission_lgb[target_submission_col] = lgb_pred.astype(float)
submission_lgb.to_csv("submissions/lightgbm_with_id_random_search.csv", index=False)

submission_xgb = sample_sub.copy()
submission_xgb[target_submission_col] = xgb_pred.astype(float)
submission_xgb.to_csv("submissions/xgboost_with_id_random_search.csv", index=False)

submission_cat = sample_sub.copy()
submission_cat[target_submission_col] = cat_pred.astype(float)
submission_cat.to_csv("submissions/catboost_with_id_random_search.csv", index=False)

print("Saved individual submissions.")

Saved individual submissions.


In [17]:
lgb_path = "submissions/lightgbm_with_id_random_search.csv"
xgb_path = "submissions/xgboost_with_id_random_search.csv"
cat_path = "submissions/catboost_with_id_random_search.csv"

lgb_sub = pd.read_csv(lgb_path)
xgb_sub = pd.read_csv(xgb_path)
cat_sub = pd.read_csv(cat_path)

In [18]:
target_col = lgb_sub.columns[1]

# 2-model blends
blend_lgb_xgb_70_30 = lgb_sub.copy()
blend_lgb_xgb_70_30[target_col] = (
    0.70 * lgb_sub[target_col] +
    0.30 * xgb_sub[target_col]
)
blend_lgb_xgb_70_30.to_csv("submissions/blend_lgb_xgb_70_30.csv", index=False)

In [19]:
blend_lgb_xgb_60_40 = lgb_sub.copy()
blend_lgb_xgb_60_40[target_col] = (
    0.60 * lgb_sub[target_col] +
    0.40 * xgb_sub[target_col]
)
blend_lgb_xgb_60_40.to_csv("submissions/blend_lgb_xgb_60_40.csv", index=False)

In [21]:
# 3-model blends
blend_lgb_xgb_cat_65_25_10 = lgb_sub.copy()
blend_lgb_xgb_cat_65_25_10[target_col] = (
    0.65 * lgb_sub[target_col] +
    0.25 * xgb_sub[target_col] +
    0.10 * cat_sub[target_col]
)
blend_lgb_xgb_cat_65_25_10.to_csv("submissions/blend_lgb_xgb_cat_65_25_10.csv", index=False)

In [22]:
blend_lgb_xgb_cat_60_25_15 = lgb_sub.copy()
blend_lgb_xgb_cat_60_25_15[target_col] = (
    0.60 * lgb_sub[target_col] +
    0.25 * xgb_sub[target_col] +
    0.15 * cat_sub[target_col]
)
blend_lgb_xgb_cat_60_25_15.to_csv("submissions/blend_lgb_xgb_cat_60_25_15.csv", index=False)

In [23]:
blend_lgb_xgb_cat_70_15_15 = lgb_sub.copy()
blend_lgb_xgb_cat_70_15_15[target_col] = (
    0.70 * lgb_sub[target_col] +
    0.15 * xgb_sub[target_col] +
    0.15 * cat_sub[target_col]
)
blend_lgb_xgb_cat_70_15_15.to_csv("submissions/blend_lgb_xgb_cat_70_15_15.csv", index=False)
